# Action → future-video attention maps (MM or full-all)

**Model:** MM (`rgb_depth_flow`, step 21700): `[R0 | R_fut | D_fut | F_fut | A]`, action attends all futures.

**Models** (`MODEL` / env `MMATT_MODEL`):
- `mm`: MM; the action reads all three futures in one pass.
- `full_all`: full layout with video loss on all futures, where the action reads **one** future (one-hot gate). Each window is run three times (`active_future` = rgb, depth, flow), and each stream's map comes from its own pass. In that pass the other two futures get exactly zero attention from the action.

**What this shows:** for each replan window, where the **action tokens** look inside each future video (RGB, depth, flow) and in R0.
1. Run `infer_joint` as in eval (same layout, mask, 10 steps); it also decodes the predicted futures.
2. At every joint self-attention call (layer × denoising step), take the attention weights of the 32 action query tokens from the model's own `q`, `k` and mask, average them over heads and action tokens, and keep the weight on **every video token**.
3. Average over the chosen layers and steps (`LAYERS`, `STEPS_SEL`), reshape each block onto its token grid (R0: 1 × 7 × 14; each future: 2 latent frames × 7 × 14; the two cameras side by side), and overlay on the frames.

**Notes:**
- The model's computation is unchanged: the attention function is wrapped from this notebook only while the model runs. No repo file is modified.
- One latent frame covers 4 video frames; maps are overlaid on the last video frame of each latent frame (frames 4 and 8).
- Colours are attention **per token**. With `NORM = "per_map"` (default, CF-WAM style) each overlay is scaled to its own 99th percentile, so the spatial pattern inside each video is visible. With `"shared"`, one scale is used across R0 and the futures, which compares intensity, but a single sink token in R0 can wash out the futures. Titles give each block's total share.

Set `QUICK = True` (or `MMATT_QUICK=1`) for a fast smoke run.

In [ ]:
import os, time, contextlib
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

REPO = "/data/mgovind/FastWAM"
os.chdir(REPO)
os.environ.setdefault("DIFFSYNTH_MODEL_BASE_PATH", f"{REPO}/checkpoints")
os.environ.setdefault("DIFFSYNTH_DOWNLOAD_SOURCE", "huggingface")

# ---------------- config ----------------
MODEL = os.environ.get("MMATT_MODEL", "mm")   # "mm" or "full_all"
MODELS = {
    # MM: action reads all three futures in one pass
    "mm": dict(task="libero_joint_2cam224_1e-4_rgb_depth_flow_modality_specific",
               run="runs/libero_joint_2cam224_1e-4_rgb_depth_flow/2026-08-28_09-13-18", step=21700),
    # full layout, video loss on all futures, action reads ONE future (one-hot gate) -> one pass per stream
    "full_all": dict(task="libero_joint_2cam224_1e-4_random_full_all",
                     run="runs/libero_joint_2cam224_1e-4_random_full_all/2026-10-12_16-35-37", step=21700),
}
TASK, RUN, STEP = MODELS[MODEL]["task"], MODELS[MODEL]["run"], MODELS[MODEL]["step"]
EPISODE = 0              # dataset episode id (0 = first spatial episode)
STRIDE = 10              # frames between windows (= eval replan_steps); full trajectory by default
N_WINDOWS = None         # None = every STRIDE frames over the whole episode; or an int = evenly spread windows
STEPS = 10               # denoising steps (= eval)
LAYERS = None            # None = all 30 layers, or e.g. range(20, 30)
STEPS_SEL = None         # None = all denoising steps, or e.g. [STEPS - 1] (last, cleanest)
SEED = 42
NORM = "per_map"         # "per_map": each overlay scaled to its own 99th percentile (CF-WAM style);
                         # "shared": one scale for R0 and all futures (compares intensity, but a sink token can dominate)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
QUICK = os.environ.get("MMATT_QUICK", "0") == "1"
if QUICK:
    N_WINDOWS, STEPS = 2, 2
OUT = "notebooks/router_analysis/outputs/mm_attention_maps" + ("" if MODEL == "mm" else f"/{MODEL}")
os.makedirs(OUT, exist_ok=True)
STREAMS = ["rgb", "depth", "flow"]
print(MODEL, TASK, DEVICE, DTYPE, "QUICK" if QUICK else "")

In [ ]:
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from fastwam.utils.video_metrics import pil_frames_to_video_tensor

with initialize_config_dir(version_base="1.3", config_dir=f"{REPO}/configs"):
    cfg = compose(config_name="train", overrides=[f"task={TASK}"])   # cached instruction context, no T5
t0 = time.time()
model = instantiate(cfg.model, model_dtype=DTYPE, device=DEVICE)
model.load_checkpoint(f"{RUN}/checkpoints/weights/step_{STEP:06d}.pt")
model = model.to(DEVICE).eval()
ds = instantiate(cfg.data.train, pretrained_norm_stats=f"{RUN}/dataset_stats.json")
if hasattr(ds, "skip_padding_as_possible"):
    ds.skip_padding_as_possible = False          # keep episode order
edi = ds.lerobot_dataset.episode_data_index
ep_a, ep_b = int(edi["from"][EPISODE]), int(edi["to"][EPISODE])
if N_WINDOWS is None:
    frames = list(range(0, ep_b - ep_a, STRIDE))                 # every replan of the trajectory
else:
    frames = np.linspace(0, ep_b - ep_a - 1, N_WINDOWS).round().astype(int).tolist()
print(f"{type(model).__name__} loaded in {time.time()-t0:.0f}s | episode {EPISODE}: {ep_b-ep_a} frames, windows at {frames}")

In [ ]:
import fastwam.models.wan22.mot as mot_mod
_orig_attn = mot_mod.flash_attention

class ActionTokenAttention:
    """Wraps mot.flash_attention; on the joint call, keeps the action rows' attention to every key token
    (mean over heads and action tokens). One record per (denoising step, layer)."""
    def __init__(self, S, A):
        self.S, self.A, self.records = S, A, []
    def __call__(self, q, k, v, num_heads, ctx_mask=None, compatibility_mode=True):
        if ctx_mask is not None and q.shape[1] == self.S and k.shape[1] == self.S:
            with torch.no_grad():
                B, S, HD = q.shape; H = int(num_heads); Dh = HD // H
                qa = q[:, -self.A:].float().view(B, self.A, H, Dh)
                scores = torch.einsum("bahd,bshd->bhas", qa, k.float().view(B, S, H, Dh)) * (Dh ** -0.5)
                m = (ctx_mask[-self.A:] if ctx_mask.ndim == 2 else ctx_mask[:, -self.A:][:, None]).to(scores.device)
                scores = scores.masked_fill(~m, float("-inf")) if m.dtype == torch.bool else scores + m.float()
                self.records.append(scores.softmax(-1).mean(dim=(0, 1, 2)).cpu().numpy())   # [S]
        return _orig_attn(q, k, v, num_heads, ctx_mask=ctx_mask, compatibility_mode=compatibility_mode)

@contextlib.contextmanager
def wrapped(rec):
    mot_mod.flash_attention = rec
    try:
        yield rec
    finally:
        mot_mod.flash_attention = _orig_attn

def grid_info(sample):
    _, T, H, W = sample["video_rgb"].shape
    ph, pw = int(model.video_expert.patch_size[1]), int(model.video_expert.patch_size[2])
    gh, gw = H // 16 // ph, W // 16 // pw            # token grid per latent frame (VAE 16x spatial)
    n_lat = 1 + (T - 1) // 4                         # VAE 4x temporal
    return dict(H=H, W=W, T=T, gh=gh, gw=gw, tpf=gh * gw, n_fut=n_lat - 1, A=int(sample["action"].shape[0]))

def _infer(sample, g, rec, active_future=None):
    kw = dict(prompt=None, input_image=sample["video_rgb"][:, 0].to(DEVICE, DTYPE),
              num_video_frames=g["T"], action_horizon=g["A"], proprio=sample["proprio"][0].to(DEVICE),
              context=sample["context"].to(DEVICE), context_mask=sample["context_mask"].to(DEVICE),
              num_inference_steps=STEPS, seed=SEED, action_attend_mode="all")
    if active_future is not None:
        kw["active_future"] = active_future
    with wrapped(rec), torch.no_grad():
        return model.infer_joint(**kw)

def run_window(sample):
    # Returns att [passes, steps, layers, S], pred {stream: [3,T,H,W]}, grid info.
    # MM: one pass (action reads all futures). full_all: one pass per stream (action reads only that one).
    g = grid_info(sample)
    S = g["tpf"] * (1 + 3 * g["n_fut"]) + g["A"]
    passes = [None] if MODEL == "mm" else STREAMS
    atts, pred = [], {}
    for af in passes:
        rec = ActionTokenAttention(S, g["A"])
        out = _infer(sample, g, rec, af)
        atts.append(np.stack(rec.records).reshape(STEPS, -1, S))
        if af is None:
            pred = {st: pil_frames_to_video_tensor(out[f"video_{st}"]) for st in STREAMS}
        else:
            pred[af] = pil_frames_to_video_tensor(out["video"])   # only the chosen future is denoised + decoded
    return np.stack(atts), pred, g

def split_maps(att_sel, g):
    """Per-token attention vector -> R0 map [1,gh,gw] and future maps [n_fut,gh,gw] per stream."""
    t, n = g["tpf"], g["n_fut"]
    o = 0
    maps = {"R0": att_sel[o:o + t].reshape(1, g["gh"], g["gw"])}; o += t
    for s in STREAMS:
        maps[s] = att_sel[o:o + n * t].reshape(n, g["gh"], g["gw"]); o += n * t
    maps["A_share"] = att_sel[o:].sum()
    return maps
print("ok")

In [ ]:
windows = []
t0 = time.time()
for i, t in enumerate(frames):
    s = ds[ep_a + t]
    att, pred, g = run_window(s)
    layers = slice(None) if LAYERS is None else list(LAYERS)
    steps = slice(None) if STEPS_SEL is None else list(STEPS_SEL)
    sel = att[:, steps][:, :, layers].mean(axis=(1, 2))   # [passes, S]
    if MODEL == "mm":
        maps = split_maps(sel[0], g)
    else:   # each stream's map from the pass where the action reads that stream; R0 / A averaged over passes
        per = [split_maps(x, g) for x in sel]
        maps = {st: per[i][st] for i, st in enumerate(STREAMS)}
        maps["R0"] = np.mean([m["R0"] for m in per], axis=0)
        maps["A_share"] = float(np.mean([m["A_share"] for m in per]))
    windows.append(dict(t=t, sample=s, att=att, pred=pred, g=g, maps=maps))
    print(f"window {i+1}/{len(frames)} (frame {t}) {time.time()-t0:.0f}s")
np.savez(f"{OUT}/ep{EPISODE}.npz", frames=np.array(frames), att=np.stack([w["att"] for w in windows]))
print("saved", f"{OUT}/ep{EPISODE}.npz")

In [ ]:
def to01(x):
    return ((x.float().clamp(-1, 1) + 1) / 2).permute(1, 2, 0).numpy()

def overlay(ax, img, amap, vmax, title):
    if NORM == "per_map":
        vmax = float(np.percentile(amap, 99)) or float(amap.max())
    H, W = img.shape[:2]
    up = F.interpolate(torch.tensor(amap)[None, None], size=(H, W), mode="bilinear", align_corners=False)[0, 0].numpy()
    ax.imshow(img); ax.imshow(up, cmap="jet", alpha=0.45, vmin=0, vmax=vmax)
    ax.set_title(title, fontsize=8); ax.axis("off")

def plot_window(w):
    g, m, s = w["g"], w["maps"], w["sample"]
    vmax = max(m["R0"].max(), *(m[k].max() for k in STREAMS))
    lat_frames = [4 * (j + 1) for j in range(g["n_fut"])]          # last video frame of each latent frame
    ncol = 1 + 2 * g["n_fut"]
    fig, axes = plt.subplots(4, ncol, figsize=(3.2 * ncol, 7.2))
    for ax in axes.ravel(): ax.axis("off")
    overlay(axes[0, 0], to01(s["video_rgb"][:, 0]), m["R0"][0], vmax, f"R0 (input)  share {m['R0'].sum():.2f}")
    for r, st in enumerate(STREAMS, start=1):
        share = m[st].sum()
        for j, fr in enumerate(lat_frames):
            pred_img = w["pred"][st][:, fr].permute(1, 2, 0).numpy()
            overlay(axes[r, 1 + j], pred_img, m[st][j], vmax, f"{st} pred, latent {j+1} (frame {fr})")
            axes[r, 1 + g['n_fut'] + j].imshow(to01(s[f"video_{st}"][:, fr])); axes[r, 1 + g['n_fut'] + j].set_title(f"{st} GT frame {fr}", fontsize=8)
        axes[r, 0].text(0.5, 0.5, f"{st} future\nshare {share:.2f}", ha="center", va="center", fontsize=11); 
    fig.suptitle(f"{MODEL} action→video attention | episode {EPISODE}, frame {w['t']} | action→action share {m['A_share']:.2f}", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{OUT}/ep{EPISODE}_frame{w['t']:04d}.png", dpi=110); plt.show()

for w in windows:
    plot_window(w)
    plt.close("all")

In [ ]:
# Episode strip: rows = R0 / RGB / depth / flow (last future latent frame), columns = windows over time
vmax = max(max(w["maps"]["R0"].max(), *(w["maps"][k].max() for k in STREAMS)) for w in windows)
fig, axes = plt.subplots(4, len(windows), figsize=(3.0 * len(windows), 6.4))
axes = np.array(axes).reshape(4, len(windows))
for c, w in enumerate(windows):
    g, m = w["g"], w["maps"]; fr = 4 * g["n_fut"]
    overlay(axes[0, c], to01(w["sample"]["video_rgb"][:, 0]), m["R0"][0], vmax, f"frame {w['t']}: R0 {m['R0'].sum():.2f}")
    for r, st in enumerate(STREAMS, start=1):
        overlay(axes[r, c], w["pred"][st][:, fr].permute(1, 2, 0).numpy(), m[st][-1], vmax, f"{st} {m[st].sum():.2f}")
fig.suptitle(f"{MODEL} action→video attention over episode {EPISODE} (futures: last latent frame; titles = block share)", fontsize=10)
plt.tight_layout(); plt.savefig(f"{OUT}/ep{EPISODE}_strip.png", dpi=110); plt.show()

In [ ]:
# Trajectory video: per window, R0 + the three futures (last latent frame) with attention, side by side.
from PIL import Image
from fastwam.utils.video_io import save_mp4

def heat(img, amap):
    vmax = float(np.percentile(amap, 99)) or float(amap.max()) if NORM == "per_map" else VMAX_ALL
    H, W = img.shape[:2]
    up = F.interpolate(torch.tensor(amap)[None, None], size=(H, W), mode="bilinear", align_corners=False)[0, 0].numpy()
    rgba = plt.get_cmap("jet")(np.clip(up / max(vmax, 1e-12), 0, 1))[..., :3]
    return (0.55 * img + 0.45 * rgba).clip(0, 1)

VMAX_ALL = max(max(w["maps"]["R0"].max(), *(w["maps"][k].max() for k in STREAMS)) for w in windows)
video = []
for w in windows:
    g, m = w["g"], w["maps"]; fr = 4 * g["n_fut"]
    tiles = [heat(to01(w["sample"]["video_rgb"][:, 0]), m["R0"][0])]
    tiles += [heat(w["pred"][st][:, fr].permute(1, 2, 0).numpy(), m[st][-1]) for st in STREAMS]
    tiles += [to01(w["sample"]["video_rgb"][:, fr])]                         # GT RGB future for reference
    video.append(Image.fromarray((np.concatenate(tiles, axis=0) * 255).astype(np.uint8)))
path = f"{OUT}/ep{EPISODE}_trajectory.mp4"
save_mp4(video, path, fps=2)
print("rows: R0 | RGB fut | depth fut | flow fut (attention, last latent frame) | GT RGB future ->", path)